# Blind Insight Statistics Core

### **A complete descriptive + inferential statistics toolkit that runs entirely on encrypted aggregate queries**

`blind_stats.BIStatsSession` computes descriptive statistics, hypothesis tests, correlations, regression summaries, and drift/monitoring reports **without ever decrypting a single record**. Every value on this page is derived from Blind Insight `aggregate` and `count` responses over an encrypted index — raw rows never leave the vault.

**How to read this notebook.** Each section demonstrates one family of statistics. Every section ends with a compact parity check: the encrypted result is compared against the same statistic computed on a plaintext fixture, confirming they match *exactly* — while the encrypted path decrypted **0 rows**. The fixture is used **only** as a source-of-truth benchmark; it is never a data source for the `BIStatsSession` calls.

**Before running:**

1. Start the Blind Proxy.
2. Upload `fixtures/fraud_train_290.json` to a dedicated BI schema (`schemas/fraud.json` is the matching JSON schema).
3. Copy `.env.example` to `.env` and fill in `BI_EMAIL`, `BI_PASSWORD`, `BI_ORG`, plus `BI_STATS_DATASET` / `BI_STATS_SCHEMA` pointing at that schema.

The target schema must contain exactly the fixture records, or the notebook stops early so comparisons are never made against the wrong dataset.

In [ ]:
from __future__ import annotations

import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display
from scipy import stats as scipy_stats

from blind_stats import (
    BILinearRegression,
    BIStatsSession,
    BlindInsightClient,
    get_demo_config,
    load_env,
    resolve_target,
)

REPO_ROOT = Path.cwd()
DEMO_CONFIG = get_demo_config()

# Swap FIXTURE_PATH for your own plaintext benchmark file to validate against real data.
FIXTURE_PATH = REPO_ROOT / DEMO_CONFIG["fixture"]
ENV_PATH = REPO_ROOT / ".env"
RISK_DOMAIN = DEMO_CONFIG["field_domains"]["risk_level"]
RISK_SUPPORT = list(range(RISK_DOMAIN[0], RISK_DOMAIN[1] + 1))

load_env(str(ENV_PATH))

records = json.loads(FIXTURE_PATH.read_text())
rows = [record["data"] for record in records]
df_fixture = pd.DataFrame(rows)
df_fixture["risk_level"] = df_fixture["risk_level"].astype(int)

risks = df_fixture["risk_level"].tolist()
fraud_types = sorted(df_fixture["fraud_type"].unique().tolist())
active_values = sorted(df_fixture["is_active"].unique().tolist())

display(Markdown(f"Loaded **{len(rows):,}** fixture records from `{FIXTURE_PATH.relative_to(REPO_ROOT)}`."))
df_fixture.head()

## Validation Helpers

These helpers are **not** used by `BIStatsSession` — they only recompute expected values from the plaintext fixture so we can prove the encrypted results match. `parity_note` renders the compact per-section verdict; the other helpers build the comparison rows and the shared table/scalar formatters used throughout the notebook.

In [ ]:
def filter_rows(source_rows: list[dict[str, Any]], *filters: str) -> list[dict[str, Any]]:
    filtered = source_rows
    for raw_filter in filters:
        for filter_part in str(raw_filter).split(","):
            if not filter_part:
                continue
            field, expected = filter_part.split(":", 1)
            filtered = [row for row in filtered if str(row[field]).lower() == expected.lower()]
    return filtered


def sample_variance(values: list[float], ddof: int = 1) -> float:
    mean = sum(values) / len(values)
    return sum((value - mean) ** 2 for value in values) / (len(values) - ddof)


def quantile_from_support(values: list[int], q: float) -> int:
    counts = Counter(values)
    rank = max(1, math.ceil(q * len(values)))
    cumulative = 0
    for value in RISK_SUPPORT:
        cumulative += counts[value]
        if cumulative >= rank:
            return value
    return RISK_SUPPORT[-1]


def close_enough(actual: float, expected: float, rel_tol: float = 1e-12, abs_tol: float = 1e-12) -> bool:
    return math.isclose(float(actual), float(expected), rel_tol=rel_tol, abs_tol=abs_tol)


def check_equal(label: str, actual: Any, expected: Any) -> dict[str, Any]:
    ok = actual == expected
    return {"check": label, "real_bi": actual, "expected_fixture": expected, "ok": ok}


def check_close(label: str, actual: float, expected: float) -> dict[str, Any]:
    ok = close_enough(actual, expected)
    return {"check": label, "real_bi": actual, "expected_fixture": expected, "ok": ok}


def show_checks(rows_: list[dict[str, Any]]) -> pd.DataFrame:
    result = pd.DataFrame(rows_)
    if not result.empty and not result["ok"].all():
        failed = result.loc[~result["ok"], ["check", "expected_fixture", "real_bi", "ok"]]
        display(Markdown(f"**Comparison mismatch:** {len(failed)} value(s) differed. Expected fixture values vs real BI values:"))
        display(failed)
    return result


def parity_note(label: str, checks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Compact per-section verdict: one line when everything matches, a table only on mismatch."""
    passed = sum(1 for c in checks if c["ok"])
    total = len(checks)
    if passed == total:
        display(Markdown(f"**✓ {label}: {total}/{total} values match plaintext exactly — computed from 0 decrypted rows.**"))
    else:
        display(Markdown(f"**✗ {label}: {passed}/{total} matched.** Mismatching values (expected fixture vs real BI):"))
        display(pd.DataFrame([c for c in checks if not c["ok"]])[["check", "expected_fixture", "real_bi"]])
    return checks


# Shared formatters for the grouped / multivariate / monitoring sections.
def grouped_result_frame(result) -> pd.DataFrame:
    table = pd.DataFrame(result.statistics).T
    table.index.name = result.group_field
    table["n"] = pd.Series(result.n)
    table["method"] = result.method
    table["exact"] = result.exact
    table["queries"] = result.queries
    return table.reset_index()


def scalar_result_row(name: str, result) -> dict[str, Any]:
    return {
        "statistic": name,
        "value": result.statistic,
        "pvalue": result.pvalue,
        "confidence_interval": result.confidence_interval,
        "n": result.n,
        "method": result.method,
        "exact": result.exact,
        "queries": result.queries,
        "warnings": "; ".join(result.warnings),
    }


def matrix_frame(result, ndigits: int = 6) -> pd.DataFrame:
    table = pd.DataFrame(result.matrix).T
    table = table[result.fields]
    return table.round(ndigits)

## Connect to the Encrypted Proxy

Create a live `BlindInsightClient`, warm up the proxy, and open a `BIStatsSession`. `field_domains` tells the session the integer domain / categorical support of each field so it can decompose statistics into range- and equality-count queries. The guard below verifies the encrypted schema holds exactly the fixture's record count before any comparison is made.

In [ ]:
target = resolve_target(DEMO_CONFIG)
org = target["org"]
dataset = target["dataset"]
schema = target["schema"]
proxy_url = target["proxy_url"]
verify_ssl = target["verify_ssl"]
client = BlindInsightClient(proxy_url=proxy_url, verify_ssl=verify_ssl)
client.warm_up(org, dataset, schema)

stats = BIStatsSession(
    client,
    org=org,
    dataset=dataset,
    schema=schema,
    field_domains={
        "risk_level": RISK_DOMAIN,
        "fraud_type": fraud_types,
        "is_active": active_values,
    },
    max_workers=4,
    retries=2,
)

actual_count = stats.count().statistic
expected_count = len(rows)
if actual_count != expected_count:
    display(
        Markdown(
            "**Record-count mismatch:** the BI schema does not match the fixture. "
            "Subsequent comparisons will show expected fixture values vs real BI values."
        )
    )
    display(pd.DataFrame([{"check": "record count", "expected_fixture": expected_count, "real_bi": actual_count, "ok": False}]))

display(Markdown(f"Connected to `{proxy_url}`. Dataset/schema: `{dataset}` / `{schema}`. Record count: **{actual_count:,}**. SSL verification: `{verify_ssl}`."))

## Reading Result Objects

Every method returns a typed `*Result` dataclass — carrying the `statistic`, any `estimate` detail, `n`, the `method` used, whether the answer is `exact`, the number of BI `queries` issued, and any `warnings`. Each dataclass implements `_repr_html_`, so a result object renders on its own with `display(...)`.

In [ ]:
count_result = stats.count()
mean_result = stats.mean("risk_level")

display(count_result)
pd.DataFrame(
    [
        vars(count_result),
        vars(mean_result),
    ],
    index=["count", "mean"],
)

## Descriptive Statistics

The core descriptive summaries. Scalars like count/sum/mean/min/max map directly onto BI `count` and numeric `aggregate` queries; variance and quantiles are computed **exactly** from per-value counts across the integer support, so they match plaintext to the last digit.

### Numeric Aggregates

`count`, `sum_`, `mean`, `min_`, `max_`, and `range_` over the encrypted index.

In [ ]:
aggregate_checks = [
    check_equal("count", stats.count().statistic, len(rows)),
    check_close("sum risk_level", stats.sum_("risk_level").statistic, float(sum(risks))),
    check_close("mean risk_level", stats.mean("risk_level").statistic, sum(risks) / len(risks)),
    check_equal("min risk_level", stats.min_("risk_level").statistic, float(min(risks))),
    check_equal("max risk_level", stats.max_("risk_level").statistic, float(max(risks))),
    check_equal("range risk_level", stats.range_("risk_level").statistic, float(max(risks) - min(risks))),
]

display(show_checks(aggregate_checks))
parity_note("Numeric aggregates", aggregate_checks)

### Variance, Quantiles, and Distribution Shape

`var`/`std` (with `ddof`), `quantile`/`median`/`iqr`, `percentile_rank`, and `ecdf`. For an integer support these are exact because they are assembled from BI counts for each support value.

In [ ]:
q25 = quantile_from_support(risks, 0.25)
q50 = quantile_from_support(risks, 0.5)
q75 = quantile_from_support(risks, 0.75)
percentile_50 = sum(1 for risk in risks if risk <= 50) / len(risks)

shape_checks = [
    check_close("sample variance", stats.var("risk_level", ddof=1).statistic, sample_variance([float(v) for v in risks], ddof=1)),
    check_close("population variance", stats.var("risk_level", ddof=0).statistic, sample_variance([float(v) for v in risks], ddof=0)),
    check_close("sample std", stats.std("risk_level", ddof=1).statistic, math.sqrt(sample_variance([float(v) for v in risks], ddof=1))),
    check_equal("q25", stats.quantile("risk_level", 0.25).statistic, q25),
    check_equal("median", stats.median("risk_level").statistic, q50),
    check_equal("q75", stats.quantile("risk_level", 0.75).statistic, q75),
    check_equal("iqr", stats.iqr("risk_level").statistic, q75 - q25),
    check_close("percentile_rank(50)", stats.percentile_rank("risk_level", 50).statistic, percentile_50),
]

display(show_checks(shape_checks))

ecdf_values = [25, 50, 75]
expected_ecdf = {str(value): sum(1 for risk in risks if risk <= value) / len(risks) for value in ecdf_values}
actual_ecdf = stats.ecdf("risk_level", ecdf_values).statistic
ecdf_table = pd.DataFrame({"real_bi": actual_ecdf, "expected_fixture": expected_ecdf})
ecdf_table["ok"] = ecdf_table["real_bi"] == ecdf_table["expected_fixture"]
display(Markdown("### ECDF at 25 / 50 / 75"))
display(ecdf_table)

parity_note("Variance, quantiles & shape", shape_checks)

### `describe` — One-Call Summary

`describe` bundles the descriptive functions into a single structured `DescribeResult`, tracking per-statistic exactness alongside the values.

In [ ]:
description = stats.describe("risk_level")
expected_description = {
    "count": len(rows),
    "mean": sum(risks) / len(risks),
    "min": float(min(risks)),
    "max": float(max(risks)),
    "range": float(max(risks) - min(risks)),
    "variance": sample_variance([float(v) for v in risks], ddof=1),
    "std": math.sqrt(sample_variance([float(v) for v in risks], ddof=1)),
    "q25": q25,
    "median": q50,
    "q75": q75,
    "iqr": q75 - q25,
}

describe_rows = []
for key, expected in expected_description.items():
    actual = description.statistics[key]
    ok = close_enough(actual, expected) if isinstance(expected, float) else actual == expected
    describe_rows.append({"stat": key, "real_bi": actual, "expected_fixture": expected, "exact": description.exact[key], "ok": ok})

describe_table = pd.DataFrame(describe_rows)
display(describe_table)
if not describe_table["ok"].all():
    failed_describe = describe_table.loc[~describe_table["ok"], ["stat", "expected_fixture", "real_bi", "ok"]]
    display(Markdown("**Describe mismatch:** expected fixture values vs real BI values:"))
    display(failed_describe)

## Filtering and Segmentation

Filter strings ride the BI encrypted search syntax, so every statistic can be scoped to a subpopulation without decryption. `where=` applies per-call filters; session-level `default_filters` are merged into every query.

### Scoped Statistics and Default Filters

In [ ]:
active_true = filter_rows(rows, "is_active:true")
active_risks = [int(row["risk_level"]) for row in active_true]

filtered_stats = BIStatsSession(
    client,
    org=org,
    dataset=dataset,
    schema=schema,
    default_filters=["is_active:true"],
    field_domains={"risk_level": RISK_DOMAIN, "fraud_type": fraud_types, "is_active": active_values},
    max_workers=4,
    retries=2,
)

fraud_type = fraud_types[0]
filtered_type_rows = filter_rows(rows, "is_active:true", f"fraud_type:{fraud_type}")

filter_checks = [
    check_equal("count where is_active:true", stats.count(where="is_active:true").statistic, len(active_true)),
    check_close(
        "mean risk_level where is_active:true",
        stats.mean("risk_level", where="is_active:true").statistic,
        sum(active_risks) / len(active_risks),
    ),
    check_equal("default filter count", filtered_stats.count().statistic, len(active_true)),
    check_equal(
        "default filter + fraud_type count",
        filtered_stats.count_eq("fraud_type", fraud_type).statistic,
        len(filtered_type_rows),
    ),
]

display(show_checks(filter_checks))
parity_note("Filters & default filters", filter_checks)

### Composing Multiple Filters

Additional filters can be supplied as a list or as a comma-separated string, and they compose on top of the session `default_filters`.

In [ ]:
multi_filter_stats = BIStatsSession(
    client,
    org=org,
    dataset=dataset,
    schema=schema,
    default_filters=["is_active:true"],
    field_domains={"risk_level": RISK_DOMAIN, "fraud_type": fraud_types, "is_active": active_values},
    max_workers=4,
    retries=2,
)

multi_fraud_type = fraud_types[0]
multi_jurisdiction = sorted(df_fixture["account_jurisdiction"].unique().tolist())[0]
multi_filters_list = [f"fraud_type:{multi_fraud_type}", f"account_jurisdiction:{multi_jurisdiction}"]
multi_filters_string = ",".join(multi_filters_list)

multi_rows = filter_rows(rows, "is_active:true", *multi_filters_list)
multi_risks = [int(row["risk_level"]) for row in multi_rows]

multi_filter_checks = [
    check_equal(
        "default is_active:true + fraud_type + jurisdiction count",
        multi_filter_stats.count(where=multi_filters_list).statistic,
        len(multi_rows),
    ),
    check_equal(
        "same filters as comma-separated string",
        multi_filter_stats.count(where=multi_filters_string).statistic,
        len(multi_rows),
    ),
]

if multi_risks:
    multi_filter_checks.extend(
        [
            check_close(
                "default + two filters mean risk_level",
                multi_filter_stats.mean("risk_level", where=multi_filters_list).statistic,
                sum(multi_risks) / len(multi_risks),
            ),
            check_equal(
                "default + two filters min risk_level",
                multi_filter_stats.min_("risk_level", where=multi_filters_list).statistic,
                float(min(multi_risks)),
            ),
            check_equal(
                "default + two filters max risk_level",
                multi_filter_stats.max_("risk_level", where=multi_filters_list).statistic,
                float(max(multi_risks)),
            ),
        ]
    )
else:
    display(
        Markdown(
            "**No fixture rows matched the selected multi-filter combination.** "
            "Choose a different fraud type or jurisdiction to demonstrate filtered aggregates."
        )
    )

display(show_checks(multi_filter_checks))
parity_note("Multiple composed filters", multi_filter_checks)

### Proportions, Rates, and Scoped Counts

`proportion` and `rate` are ratios of BI count queries; `count_range` and `count_eq` are the range/equality primitives underneath.

In [ ]:
active_count = len(active_true)
type_count = len(filter_rows(rows, f"fraud_type:{fraud_type}"))
type_active_count = len(filter_rows(rows, f"fraud_type:{fraud_type}", "is_active:true"))

rate_checks = [
    check_equal("count_range risk 0~49", stats.count_range("risk_level", 0, 49).statistic, sum(1 for r in risks if 0 <= r <= 49)),
    check_equal("count_eq fraud_type", stats.count_eq("fraud_type", fraud_type).statistic, type_count),
    check_close("proportion is_active:true", stats.proportion("is_active:true").statistic, active_count / len(rows)),
    check_close(
        "rate is_active:true within fraud_type",
        stats.rate("is_active:true", denominator_filter=f"fraud_type:{fraud_type}").statistic,
        type_active_count / type_count,
    ),
]

display(show_checks(rate_checks))
parity_note("Proportions, rates & scoped counts", rate_checks)

## Distributions: Histograms, Frequency, and Mode

`histogram` uses BI range-count aggregates over user-defined bins; `frequency` and `mode` use one equality-count per category. The counts are exact.

In [ ]:
bins = [(0, 24), (25, 49), (50, 74), (75, 102)]
hist = stats.histogram("risk_level", bins=bins)
expected_hist = {f"{low}~{high}": sum(1 for risk in risks if low <= risk <= high) for low, high in bins}

freq = stats.frequency("fraud_type")
expected_counts_counter = Counter(row["fraud_type"] for row in rows)
expected_freq = {value: expected_counts_counter[value] for value in fraud_types}
expected_mode = max(expected_freq.items(), key=lambda item: item[1])[0]

display(Markdown("### Risk Histogram"))
hist_table = pd.DataFrame({"real_bi": hist.counts, "expected_fixture": expected_hist})
hist_table["ok"] = hist_table["real_bi"] == hist_table["expected_fixture"]
display(hist_table)

display(Markdown("### Fraud Type Frequency"))
freq_table = pd.DataFrame(
    {
        "bi_count": freq.counts,
        "fixture_count": expected_freq,
        "bi_proportion": freq.proportions,
    }
)
freq_table["ok"] = freq_table["bi_count"] == freq_table["fixture_count"]
display(freq_table)

actual_mode = stats.mode("fraud_type").statistic
display(Markdown(f"Expected mode from fixture: **{expected_mode}**. Real BI mode: **{actual_mode}**."))

distribution_checks = [
    check_equal("histogram bucket counts", dict(hist.counts), expected_hist),
    check_equal("frequency counts", dict(freq.counts), expected_freq),
    check_equal("mode fraud_type", actual_mode, expected_mode),
]
parity_note("Histograms, frequency & mode", distribution_checks)

## Categorical Tables and Proportion Tests

These tests are built purely from BI count queries. SciPy appears only to compute the plaintext benchmark p-values and exact distribution math over the aggregate count tables — never over rows.

### Two-Way Crosstab

`crosstab` assembles a contingency table (with row/column totals) from equality counts.

In [ ]:
crosstab_result = stats.crosstab("fraud_type", "is_active")
fixture_crosstab = (
    pd.crosstab(df_fixture["fraud_type"], df_fixture["is_active"])
    .reindex(index=fraud_types, columns=active_values, fill_value=0)
    .astype(int)
)
bi_crosstab = pd.DataFrame(crosstab_result.counts).T.reindex(index=fraud_types, columns=active_values).astype(int)

crosstab_compare = pd.concat(
    {"real_bi": bi_crosstab, "plaintext_fixture": fixture_crosstab},
    axis=1,
)
crosstab_compare[("comparison", "ok")] = (bi_crosstab == fixture_crosstab).all(axis=1)

display(Markdown("### Crosstab: fraud_type x is_active"))
display(crosstab_compare)

crosstab_checks = [
    check_equal("crosstab fraud_type x is_active counts", bi_crosstab.to_dict(), fixture_crosstab.to_dict()),
    check_equal("crosstab row totals", crosstab_result.row_totals, fixture_crosstab.sum(axis=1).to_dict()),
    check_equal("crosstab column totals", crosstab_result.col_totals, fixture_crosstab.sum(axis=0).to_dict()),
]
parity_note("Crosstab", crosstab_checks)

### Chi-Square Tests

Goodness-of-fit (`chisquare`) on one categorical field, and independence (`chi2_independence`) on the crosstab.

In [ ]:
bi_chisquare = stats.chisquare("fraud_type")
fixture_freq = fixture_crosstab.sum(axis=1).reindex(fraud_types).astype(int)
fixture_chisquare = scipy_stats.chisquare(fixture_freq.to_list())

bi_chi2 = stats.chi2_independence("fraud_type", "is_active")
fixture_chi2_stat, fixture_chi2_p, fixture_chi2_dof, fixture_chi2_expected = scipy_stats.chi2_contingency(
    fixture_crosstab.to_numpy()
)

chi_checks = [
    check_close("chi-square goodness statistic", bi_chisquare.statistic, fixture_chisquare.statistic),
    check_close("chi-square goodness p-value", bi_chisquare.pvalue, fixture_chisquare.pvalue),
    check_close("chi-square independence statistic", bi_chi2.statistic, fixture_chi2_stat),
    check_close("chi-square independence p-value", bi_chi2.pvalue, fixture_chi2_p),
    check_equal("chi-square independence dof", bi_chi2.estimate["dof"], int(fixture_chi2_dof)),
]

display(pd.DataFrame(
    [
        {"function": "chisquare", "real_bi": bi_chisquare.statistic, "plaintext_fixture": fixture_chisquare.statistic,
         "pvalue_bi": bi_chisquare.pvalue, "pvalue_plaintext": fixture_chisquare.pvalue},
        {"function": "chi2_independence", "real_bi": bi_chi2.statistic, "plaintext_fixture": fixture_chi2_stat,
         "pvalue_bi": bi_chi2.pvalue, "pvalue_plaintext": fixture_chi2_p},
    ]
))
parity_note("Chi-square tests", chi_checks)

### 2×2 Association: Fisher Exact, Odds Ratio, Relative Risk

Exact 2×2 measures computed on two concrete fraud types crossed with `is_active`.

In [ ]:
binary_fraud_types = fraud_types[:2]
positive_active = "true"
negative_active = "false"
fixture_2x2 = fixture_crosstab.loc[binary_fraud_types, [positive_active, negative_active]].to_numpy()

bi_fisher = stats.fisher_exact(
    "fraud_type",
    "is_active",
    row_values=binary_fraud_types,
    col_values=[positive_active, negative_active],
)
fixture_fisher = scipy_stats.fisher_exact(fixture_2x2)

bi_odds = stats.odds_ratio(
    "fraud_type",
    "is_active",
    exposed=binary_fraud_types[0],
    unexposed=binary_fraud_types[1],
    outcome=positive_active,
    nonoutcome=negative_active,
)
bi_risk = stats.relative_risk(
    "fraud_type",
    "is_active",
    exposed=binary_fraud_types[0],
    unexposed=binary_fraud_types[1],
    outcome=positive_active,
    nonoutcome=negative_active,
)

a, b = fixture_2x2[0]
c, d = fixture_2x2[1]
fixture_odds = (a * d) / (b * c) if b * c else math.inf if a * d else math.nan
fixture_risk = (a / (a + b)) / (c / (c + d)) if (a + b) and c else math.nan

twoby2_checks = [
    check_close("Fisher exact odds ratio", bi_fisher.statistic, fixture_fisher.statistic),
    check_close("Fisher exact p-value", bi_fisher.pvalue, fixture_fisher.pvalue),
    check_close("odds ratio", bi_odds.statistic, fixture_odds),
    check_close("relative risk", bi_risk.statistic, fixture_risk),
]

display(pd.DataFrame(
    [
        scalar_result_row("fisher_exact", bi_fisher),
        scalar_result_row("odds_ratio", bi_odds),
        scalar_result_row("relative_risk", bi_risk),
    ]
))
parity_note("Fisher / odds ratio / relative risk", twoby2_checks)

### Proportion z-Tests and Confidence Intervals

One- and two-proportion z-tests, plus Wilson and Clopper-Pearson confidence intervals for a proportion.

In [ ]:
active_successes = int((df_fixture["is_active"] == positive_active).sum())
active_trials = len(df_fixture)
active_prop = active_successes / active_trials
null_prop = 0.5
one_prop_se = math.sqrt(null_prop * (1 - null_prop) / active_trials)
fixture_z_one = (active_prop - null_prop) / one_prop_se
fixture_z_one_p = 2 * scipy_stats.norm.sf(abs(fixture_z_one))
bi_z_one = stats.ztest_proportion("is_active:true", p=null_prop)

fixture_group_counts = fixture_crosstab.loc[binary_fraud_types, [positive_active, negative_active]]
successes_a = int(fixture_group_counts.loc[binary_fraud_types[0], positive_active])
trials_a = int(fixture_group_counts.loc[binary_fraud_types[0]].sum())
successes_b = int(fixture_group_counts.loc[binary_fraud_types[1], positive_active])
trials_b = int(fixture_group_counts.loc[binary_fraud_types[1]].sum())
prop_a = successes_a / trials_a
prop_b = successes_b / trials_b
pooled = (successes_a + successes_b) / (trials_a + trials_b)
two_prop_se = math.sqrt(pooled * (1 - pooled) * ((1 / trials_a) + (1 / trials_b)))
fixture_z_two = (prop_a - prop_b) / two_prop_se
fixture_z_two_p = 2 * scipy_stats.norm.sf(abs(fixture_z_two))
bi_z_two = stats.ztest_proportions("is_active:true", "fraud_type", binary_fraud_types[0], binary_fraud_types[1])

bi_wilson = stats.proportion_ci("is_active:true", method="wilson")
bi_clopper = stats.proportion_ci("is_active:true", method="clopper-pearson")

alpha = 0.05
z = scipy_stats.norm.ppf(1 - alpha / 2)
wilson_den = 1 + z**2 / active_trials
wilson_center = (active_prop + z**2 / (2 * active_trials)) / wilson_den
wilson_half = z * math.sqrt((active_prop * (1 - active_prop) + z**2 / (4 * active_trials)) / active_trials) / wilson_den
fixture_wilson = (max(0.0, wilson_center - wilson_half), min(1.0, wilson_center + wilson_half))
fixture_clopper = (
    0.0 if active_successes == 0 else scipy_stats.beta.ppf(alpha / 2, active_successes, active_trials - active_successes + 1),
    1.0
    if active_successes == active_trials
    else scipy_stats.beta.ppf(1 - alpha / 2, active_successes + 1, active_trials - active_successes),
)

prop_checks = [
    check_close("one-proportion z statistic", bi_z_one.statistic, fixture_z_one),
    check_close("one-proportion z p-value", bi_z_one.pvalue, fixture_z_one_p),
    check_close("two-proportion z statistic", bi_z_two.statistic, fixture_z_two),
    check_close("two-proportion z p-value", bi_z_two.pvalue, fixture_z_two_p),
    check_close("Wilson CI lower", bi_wilson.confidence_interval[0], fixture_wilson[0]),
    check_close("Wilson CI upper", bi_wilson.confidence_interval[1], fixture_wilson[1]),
    check_close("Clopper-Pearson CI lower", bi_clopper.confidence_interval[0], fixture_clopper[0]),
    check_close("Clopper-Pearson CI upper", bi_clopper.confidence_interval[1], fixture_clopper[1]),
]

display(pd.DataFrame(
    [
        scalar_result_row("ztest_proportion", bi_z_one),
        scalar_result_row("ztest_proportions", bi_z_two),
        scalar_result_row("proportion_ci (Wilson)", bi_wilson),
        scalar_result_row("proportion_ci (Clopper-Pearson)", bi_clopper),
    ]
))

categorical_checks = crosstab_checks + chi_checks + twoby2_checks + prop_checks
parity_note("Proportion z-tests & confidence intervals", prop_checks)

## Grouped Statistics and Mean Inference

Grouped numeric summaries and t-based mean inference. Each group's statistics are computed from scoped aggregate/count queries; the tables below show the structured result objects returned by `stats.py`.

### Grouped Numeric Summaries

`groupby_count`, `groupby_mean`, `groupby_var`, `groupby_std` split by a categorical field.

In [ ]:
grouped_count = stats.groupby_count("is_active")
grouped_mean = stats.groupby_mean("risk_level", "is_active")
grouped_var = stats.groupby_var("risk_level", "is_active")
grouped_std = stats.groupby_std("risk_level", "is_active")

phase3_grouped_tables = {
    "groupby_count(is_active)": grouped_result_frame(grouped_count),
    "groupby_mean(risk_level, by=is_active)": grouped_result_frame(grouped_mean),
    "groupby_var(risk_level, by=is_active)": grouped_result_frame(grouped_var),
    "groupby_std(risk_level, by=is_active)": grouped_result_frame(grouped_std),
}

for title, table in phase3_grouped_tables.items():
    display(Markdown(f"### `{title}`"))
    display(table)

### Mean Intervals and t-Tests

`mean_ci`, one-sample `ttest_1samp`, and two-sample `ttest_ind` in both Welch and pooled variants.

In [ ]:
mean_interval = stats.mean_ci("risk_level", confidence_level=0.95)
one_sample_t = stats.ttest_1samp("risk_level", popmean=50.0)
welch_t = stats.ttest_ind("risk_level", "is_active", "false", "true")
pooled_t = stats.ttest_ind("risk_level", "is_active", "false", "true", equal_var=True)

display(pd.DataFrame(
    [
        scalar_result_row("mean_ci(risk_level)", mean_interval),
        scalar_result_row("ttest_1samp(risk_level, popmean=50)", one_sample_t),
        scalar_result_row("ttest_ind(..., equal_var=False)", welch_t),
        scalar_result_row("ttest_ind(..., equal_var=True)", pooled_t),
    ]
))

### Multi-Group Mean Tests (ANOVA)

`anova_oneway` and `welch_anova` across the configured fraud-type categories.

In [ ]:
one_way = stats.anova_oneway("risk_level", "fraud_type", groups=fraud_types)
welch_one_way = stats.welch_anova("risk_level", "fraud_type", groups=fraud_types)

display(pd.DataFrame(
    [
        scalar_result_row("anova_oneway(risk_level, by=fraud_type)", one_way),
        scalar_result_row("welch_anova(risk_level, by=fraud_type)", welch_one_way),
    ]
))

inference_checks = [
    check_equal("groupby_count groups", sorted(grouped_count.n), active_values),
    check_equal("groupby_count total", sum(grouped_count.n.values()), actual_count),
    check_equal("groupby_mean groups", sorted(grouped_mean.statistics), active_values),
    check_equal("groupby_var groups", sorted(grouped_var.statistics), active_values),
    check_equal("groupby_std groups", sorted(grouped_std.statistics), active_values),
    check_equal("mean_ci method", mean_interval.method, "t_mean_interval"),
    check_equal("one-sample t method", one_sample_t.method, "one_sample_ttest"),
    check_equal("Welch t method", welch_t.method, "welch_two_sample_ttest"),
    check_equal("pooled t method", pooled_t.method, "pooled_two_sample_ttest"),
    check_equal("ANOVA method", one_way.method, "one_way_anova"),
    check_equal("Welch ANOVA method", welch_one_way.method, "welch_anova"),
    check_equal(
        "mean/ANOVA p-values finite",
        all(math.isfinite(v) for v in [one_sample_t.pvalue, welch_t.pvalue, pooled_t.pvalue, one_way.pvalue, welch_one_way.pvalue]),
        True,
    ),
]
parity_note("Grouped statistics & mean inference", inference_checks)

## Nonparametric and Rank-Based Tests

`ks_2samp` and `mannwhitneyu` computed exactly from integer-support count vectors for each group — no row-level ranking required.

In [ ]:
ks_result = stats.ks_2samp("risk_level", "is_active", "false", "true")
mann_whitney = stats.mannwhitneyu("risk_level", "is_active", "false", "true")

display(pd.DataFrame(
    [
        scalar_result_row("ks_2samp(risk_level, by=is_active)", ks_result),
        scalar_result_row("mannwhitneyu(risk_level, by=is_active)", mann_whitney),
    ]
))

nonparam_checks = [
    check_equal("KS method", ks_result.method, "support_counts_ks_2samp"),
    check_equal("Mann-Whitney method", mann_whitney.method, "support_counts_mannwhitneyu"),
    check_equal("KS/Mann-Whitney p-values finite", all(math.isfinite(v) for v in [ks_result.pvalue, mann_whitney.pvalue]), True),
]
parity_note("Nonparametric rank-based tests", nonparam_checks)

## Correlation, Multivariate Summaries, and Regression

Aggregate-only multivariate summaries. One-hot covariance/correlation use count-only marginals and pair counts (exact); binned numeric correlations use 2-D range-count tables (approximate); and regression inference requires derived moment fields (e.g. `x_times_y`, `x_squared`) to exist in the encrypted schema.

### One-Hot Covariance and Correlation

In [ ]:
onehot_fields = {"fraud_type": fraud_types[:3], "is_active": active_values}
onehot_cov = stats.onehot_covariance(onehot_fields)
onehot_corr = stats.onehot_correlation(onehot_fields)

display(Markdown("### One-Hot Covariance"))
display(matrix_frame(onehot_cov))
display(Markdown("### One-Hot Correlation"))
display(matrix_frame(onehot_corr))

### Binned Numeric Correlation and Effect Sizes

`binned_pearson`/`binned_spearman` over range bins, plus `point_biserial` and `eta_squared` effect sizes.

In [ ]:
risk_bins = [(0, 24), (25, 49), (50, 74), (75, 102)]
month_bins = [(1, 3), (4, 6), (7, 9), (10, 12)]
binned_pearson = stats.binned_pearson("risk_level", "month", x_bins=risk_bins, y_bins=month_bins)
binned_spearman = stats.binned_spearman("risk_level", "month", x_bins=risk_bins, y_bins=month_bins)

point_biserial = stats.point_biserial("risk_level", "is_active", "true", support=RISK_SUPPORT)
eta_risk_by_fraud = stats.eta_squared("risk_level", "fraud_type", groups=fraud_types[:3], support=RISK_SUPPORT)

display(Markdown("### Binned Numeric Correlations"))
display(pd.DataFrame([scalar_result_row("binned_pearson(risk_level, month)", binned_pearson), scalar_result_row("binned_spearman(risk_level, month)", binned_spearman)]))

display(Markdown("### Effect Sizes"))
display(pd.DataFrame([scalar_result_row("point_biserial(risk_level, is_active=true)", point_biserial), scalar_result_row("eta_squared(risk_level, by=fraud_type[:3])", eta_risk_by_fraud)]))

### Regression and Feature Screening

`BILinearRegression` fits from aggregate sufficient statistics, so it *requires* derived moment fields in the schema — the guard below shows the clear error when they are absent. `feature_screening` produces a per-feature report (missingness, univariate stats, association metrics) in one call.

In [ ]:
try:
    BILinearRegression().fit(stats, y="risk_level", x=["month"], moment_fields={})
except ValueError as exc:
    regression_requirement = str(exc)
else:
    regression_requirement = "unexpectedly fitted without derived moment fields"

feature_report = stats.feature_screening(
    numeric_fields=["risk_level"],
    categorical_fields=["fraud_type"],
    target_field="is_active",
    target_positive="true",
    supports={"risk_level": RISK_SUPPORT},
    values={"fraud_type": fraud_types[:3], "is_active": active_values},
)

display(Markdown("### Regression Moment-Field Requirement"))
display(Markdown(f"`BILinearRegression` correctly requires derived aggregate moment fields: `{regression_requirement}`"))

display(Markdown("### Feature Screening Report"))
display(pd.DataFrame(feature_report.statistics["numeric"]).T)
display(pd.DataFrame(feature_report.statistics["categorical"]).T)

multivariate_checks = [
    check_equal("onehot covariance exact", onehot_cov.exact, True),
    check_equal("onehot correlation exact", onehot_corr.exact, True),
    check_equal("onehot covariance n", onehot_cov.n, actual_count),
    check_equal("binned pearson approximate", binned_pearson.exact, False),
    check_equal("binned spearman approximate", binned_spearman.exact, False),
    check_equal("point-biserial finite", math.isfinite(point_biserial.statistic), True),
    check_equal("eta squared bounded", 0 <= eta_risk_by_fraud.statistic <= 1, True),
    check_equal("regression requires moment fields", "missing derived moment field" in regression_requirement, True),
    check_equal("feature report has numeric risk_level", "risk_level" in feature_report.statistics["numeric"], True),
    check_equal("feature report has fraud_type", "fraud_type" in feature_report.statistics["categorical"], True),
]
parity_note("Correlation, multivariate & regression", multivariate_checks)

## Monitoring, Drift, and Data Quality

The same BI primitives power ongoing monitoring: missingness/domain checks and outlier counts, distribution-drift metrics, period-grouped summaries, and a consolidated data-quality report.

### Missingness, Domain Violations, and Outliers

In [ ]:
missingness = stats.missingness_rate("fraud_type", ["missing"])
risk_domain_violations = stats.domain_violation_count("risk_level", domain=RISK_DOMAIN)
zscore_outliers = stats.zscore_outlier_count("risk_level", z=3.0, support=RISK_SUPPORT)
iqr_outliers = stats.iqr_outlier_count("risk_level", support=RISK_SUPPORT)

display(pd.DataFrame(
    [
        scalar_result_row("missingness_rate(fraud_type)", missingness),
        scalar_result_row("domain_violation_count(risk_level)", risk_domain_violations),
        scalar_result_row("zscore_outlier_count(risk_level, z=3)", zscore_outliers),
        scalar_result_row("iqr_outlier_count(risk_level)", iqr_outliers),
    ]
))

### Distribution Drift: PSI, Divergence, and Chi-Square

Compare the `is_active:false` baseline against the `is_active:true` current population over the `fraud_type` distribution.

In [ ]:
freq_distribution = stats.distribution("fraud_type", values=fraud_types)
psi_active = stats.population_stability_index("fraud_type", "is_active:false", "is_active:true", values=fraud_types)
js_active = stats.distribution_divergence("fraud_type", "is_active:false", "is_active:true", values=fraud_types, metric="js")
kl_active = stats.distribution_divergence("fraud_type", "is_active:false", "is_active:true", values=fraud_types, metric="kl")
drift_active = stats.drift_chi2("fraud_type", baseline_filter="is_active:false", current_filter="is_active:true", values=fraud_types)

display(Markdown("### Drift Scalars"))
display(pd.DataFrame(
    [
        scalar_result_row("population_stability_index(fraud_type)", psi_active),
        scalar_result_row("JS divergence(fraud_type)", js_active),
        scalar_result_row("KL divergence(fraud_type)", kl_active),
        scalar_result_row("drift_chi2(fraud_type)", drift_active),
    ]
))

display(Markdown("### Distribution"))
display(pd.DataFrame({"count": freq_distribution.statistic["counts"], "proportion": freq_distribution.statistic["proportions"]}))

### Period Summaries and Data-Quality Report

`period_counts`/`period_summary` roll up by a period field; `data_quality_report` bundles missingness, domain, and value checks into one suppression-aware report object.

In [ ]:
period_count_table = stats.period_counts("month", [1, 2, 3])
period_mean_table = stats.period_summary("risk_level", "month", [1, 2, 3], agg="mean")
quality_report = stats.data_quality_report(
    ["risk_level", "fraud_type"],
    missing_values={"fraud_type": ["missing"]},
    domains={"risk_level": RISK_DOMAIN},
    values={"fraud_type": fraud_types},
    suppression_policy="warn",
)

display(Markdown("### Period Counts And Means"))
display(grouped_result_frame(period_count_table))
display(grouped_result_frame(period_mean_table))

display(Markdown("### Data Quality Report"))
display(pd.DataFrame(quality_report.statistics).T)

monitoring_checks = [
    check_equal("missingness finite", math.isfinite(missingness.statistic), True),
    check_equal("risk domain violations zero", risk_domain_violations.statistic, 0),
    check_equal("zscore outlier count finite", math.isfinite(zscore_outliers.statistic), True),
    check_equal("iqr outlier count finite", math.isfinite(iqr_outliers.statistic), True),
    check_equal("distribution count total", sum(freq_distribution.statistic["counts"].values()), actual_count),
    check_equal("PSI finite", math.isfinite(psi_active.statistic), True),
    check_equal("JS finite", math.isfinite(js_active.statistic), True),
    check_equal("KL finite", math.isfinite(kl_active.statistic), True),
    check_equal("drift chi-square finite", math.isfinite(drift_active.statistic), True),
    check_equal("period count groups", sorted(period_count_table.statistics), ["1", "2", "3"]),
    check_equal("period mean groups", sorted(period_mean_table.statistics), ["1", "2", "3"]),
    check_equal("quality report has risk_level", "risk_level" in quality_report.statistics, True),
]
parity_note("Monitoring, drift & data quality", monitoring_checks)

## Privacy: Cell Suppression and Guardrails

Small cells can leak information, so `BIStatsSession` carries a `min_cell_size` policy. With `suppression_policy="warn"` it reports warnings; with `"suppress"` it masks under-threshold cells (`None`). The session also raises clear errors when a field's domain or support is missing, preventing silently wrong statistics.

In [ ]:
warning_stats = BIStatsSession(
    client,
    org=org,
    dataset=dataset,
    schema=schema,
    field_domains={"risk_level": RISK_DOMAIN, "fraud_type": fraud_types, "is_active": active_values},
    min_cell_size=1_000,
    max_workers=4,
    retries=2,
)

warning_freq = warning_stats.frequency("fraud_type")
display(Markdown("### Min-Cell-Size Warnings"))
display(pd.DataFrame({"warnings": warning_freq.warnings[:5]}))

suppressed_freq = warning_stats.frequency("fraud_type", suppression_policy="suppress")
display(Markdown("### Suppression Policy (`suppress`)"))
display(pd.DataFrame({"suppressed_count": suppressed_freq.counts, "proportion": suppressed_freq.proportions}))

display(Markdown("### Clear Errors On Missing Domain / Support"))
try:
    stats.mean("unknown_numeric_field")
except ValueError as exc:
    display(Markdown(f"Missing domain error: `{exc}`"))

try:
    stats.var("reporting_bank_id")
except ValueError as exc:
    display(Markdown(f"Missing support error: `{exc}`"))

privacy_checks = [
    check_equal("min-cell warnings emitted", bool(warning_freq.warnings), True),
    check_equal("suppression masked cells", any(value is None for value in suppressed_freq.counts.values()), True),
]
parity_note("Privacy guardrails", privacy_checks)

## Summary

Every capability above was computed from Blind Insight aggregate/count queries and then compared against the same statistic computed on the plaintext fixture. The cell below concatenates all comparisons into one final verdict.

In [ ]:
all_summary_checks = (
    aggregate_checks
    + shape_checks
    + distribution_checks
    + filter_checks
    + multi_filter_checks
    + rate_checks
    + categorical_checks
    + inference_checks
    + nonparam_checks
    + multivariate_checks
    + monitoring_checks
    + privacy_checks
)
summary = show_checks(all_summary_checks)
display(summary)
if summary["ok"].all():
    display(Markdown(f"**All {len(all_summary_checks)} real-proxy statistics checks passed — every value matched plaintext exactly, computed from 0 decrypted rows.**"))
else:
    display(Markdown("**Some real-proxy statistics checks differed. See expected values vs real BI values above.**"))